## Setup, Imports, and Triton Fixes

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

In [ ]:
import datasets
import trl
print("Successfully imported datasets version:", datasets.__version__)
print("Successfully imported trl version:", trl.__version__)

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

In [ ]:
import os
import sys
import stat
import shutil
import gc
import zipfile
import types
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# --- Prevent CUDA memory fragmentation (must be set before first CUDA alloc) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Block Mamba3 import chain (pulls cutlass/quack which aren't installed) ---
# The model only uses Mamba2. Mamba3 is imported by mamba_ssm.__init__ but never used.
# Pre-registering these in sys.modules prevents Python from loading the real files.
_mock_class = type("_Mock", (), {})
for _name, _attrs in [
    ("mamba_ssm.modules.mamba3", {"Mamba3": _mock_class}),
    ("mamba_ssm.ops.cute", {}),
    ("mamba_ssm.ops.cute.mamba3", {}),
    ("mamba_ssm.ops.cute.mamba3.mamba3_step_fn", {"mamba3_step_fn": lambda *a, **kw: None}),
]:
    if _name not in sys.modules:
        _m = types.ModuleType(_name)
        _m.__path__ = []
        for _k, _v in _attrs.items():
            setattr(_m, _k, _v)
        sys.modules[_name] = _m

# --- Kaggle / Triton Environment Fixes ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst

# --- Hyperparameters ---
SUBSAMPLE_SIZE = 600
LORA_RANK = 32
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 2e-4

OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- GRPO Hyperparameters (conservative to avoid OOM) ---
GRPO_LR = 5e-6
GRPO_EPOCHS = 1
GRPO_SAMPLES = 800          # 800 samples for GRPO
GRPO_NUM_GEN = 2             # 2 generations (was 4 → OOM)
GRPO_MAX_COMPLETION = 512    # 512 tokens max (was 1024 → OOM)
GRPO_MAX_PROMPT = 1024       # keep prompts full length

## Data Loading & Formatting

In [ ]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# ===== Stage 1: Load typed-CoT training data =====
# This CSV has columns: id, prompt, answer, thinking
# - 5 types have concise programmatic CoT in "thinking" column (300-600 chars)
# - symbol type has empty "thinking" (answer-only format, same as V2)
cot_df = pl.read_csv('/kaggle/input/prog-cot-training-data/sft_typed_cot_600.csv')
print(f"CoT training data: {len(cot_df)} samples")
print(f"Columns: {cot_df.columns}")
print(f"Has thinking: {cot_df.filter(pl.col('thinking').str.len_chars() > 0).shape[0]}")
print(f"Answer-only:  {cot_df.filter(pl.col('thinking').str.len_chars() == 0).shape[0]}")

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(cot_df.to_pandas())

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SUFFIX = "\nPut your final answer inside \\boxed{}."

def build_training_text(example):
    """Build training text with type-specific CoT in <think> blocks."""
    prompt = example["prompt"]
    answer = example["answer"]
    thinking = example.get("thinking", "") or ""
    
    user_msg = prompt + SUFFIX
    
    if thinking.strip():
        # CoT types: put concise reasoning in <think> block
        assistant_msg = f"<think>\n{thinking}\n</think>\n\\boxed{{{answer}}}"
    else:
        # Symbol / answer-only: empty thinking (same as V2)
        assistant_msg = f"\\boxed{{{answer}}}"

    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=True
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n<think>\n{thinking}\n</think>\n\\boxed{{{answer}}}<|im_end|>"
        )
    return {"text": text}

In [ ]:
hf_dataset = hf_dataset.map(
    build_training_text,
    remove_columns=hf_dataset.column_names # <--- This deletes 'id', 'prompt', and 'answer'
)

print(f"Dataset formatted. Example:\n{hf_dataset[0]['text'][:500]}")

## Model Loading & LoRA Setup

In [ ]:
# Load Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    device_map="auto", 
    trust_remote_code=True, 
    dtype=torch.bfloat16
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Force slow path — bypass the broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

# Setup LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules="all-linear", # Targets all linear layers for better reasoning performance
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Training with SFTTrainer

In [ ]:
import os
import triton.backends.nvidia.compiler as nv_compiler

# Tell Triton's environment parser where the writable Blackwell binary is
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# --- Validate that response tokens survive truncation ---
_resp_marker = "<|im_start|>assistant\n"
_truncated = 0
for ex in hf_dataset:
    toks = tokenizer(ex["text"], truncation=True, max_length=MAX_SEQ_LEN)
    decoded = tokenizer.decode(toks["input_ids"])
    if _resp_marker not in decoded:
        _truncated += 1
print(f"⚠️ {_truncated}/{len(hf_dataset)} samples lose response after truncation to {MAX_SEQ_LEN} tokens")
if _truncated > len(hf_dataset) * 0.1:
    print("WARNING: >10% samples truncated! Consider increasing MAX_SEQ_LEN")

# Control max sequence length via tokenizer (compatible with all TRL versions)
tokenizer.model_max_length = MAX_SEQ_LEN

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,           
    gradient_accumulation_steps=GRAD_ACCUM,  
    num_train_epochs=NUM_EPOCHS,             
    learning_rate=LR,                        
    logging_steps=5,                         
    bf16=True,                               
    max_grad_norm=1.0,                       
    optim="adamw_torch",                     
    lr_scheduler_type="cosine",              
    warmup_ratio=0.1,                        
    save_strategy="no",                      
    report_to="none",
    dataset_text_field="text",               
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()

## Phase 2: GRPO Reinforcement Learning

SFT gives the model format awareness. GRPO teaches it to maximize correctness under sampling randomness (temperature=1.0).
- Same LoRA adapter continues training (SFT → GRPO, single adapter output)
- LR 40x lower than SFT to preserve SFT gains
- beta=0: no reference model needed (saves ~30GB VRAM)

In [ ]:
# --- Free SFT trainer memory ---
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after SFT cleanup: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# --- Mock missing optional deps that TRL imports at module level ---
import types as _types

def _create_mock_module(name, attrs=None):
    mod = _types.ModuleType(name)
    mod.__path__ = []
    if attrs:
        for k, v in attrs.items():
            setattr(mod, k, v)
    return mod

_mock_class = type("_Mock", (), {})

_all_mocks = {
    "mergekit": {},
    "mergekit.config": {"MergeConfiguration": _mock_class},
    "mergekit.merge": {"MergeOptions": _mock_class, "run_merge": lambda *a, **kw: None},
    "mergekit.architecture": {},
    "mergekit.io": {},
    "mergekit.io.tasks": {},
    "mergekit.io.lazy_tensor_loader": {},
    "mergekit.common": {},
    "mergekit.graph": {},
    "mergekit.merge_methods": {},
    "mergekit.options": {},
    "mergekit.plan": {},
    "mergekit.sparsify": {},
    "llm_blender": {"Blender": _mock_class},
    "weave": {"EvaluationLogger": _mock_class},
    "weave.trace": {},
    "weave.trace.context": {"weave_client_context": _mock_class},
    "liger_kernel": {},
    "liger_kernel.transformers": {},
}

for pkg_name, attrs in _all_mocks.items():
    if pkg_name not in sys.modules:
        sys.modules[pkg_name] = _create_mock_module(pkg_name, attrs)

sys.modules["weave"].trace = sys.modules["weave.trace"]
sys.modules["weave.trace"].context = sys.modules["weave.trace.context"]
sys.modules["mergekit"].config = sys.modules["mergekit.config"]
sys.modules["mergekit"].merge = sys.modules["mergekit.merge"]
sys.modules["mergekit"].io = sys.modules["mergekit.io"]
sys.modules["mergekit.io"].tasks = sys.modules["mergekit.io.tasks"]
sys.modules["mergekit.io"].lazy_tensor_loader = sys.modules["mergekit.io.lazy_tensor_loader"]

print(f"✓ Mocked {len(_all_mocks)} optional TRL dependencies")

# --- GRPO imports ---
import re
from trl import GRPOTrainer, GRPOConfig

# --- Prepare GRPO dataset ---
METRIC_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

grpo_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

# --- Classify type from prompt text (train.csv has no "type" column) ---
_type_keywords = {
    'bit_ops': ['bitwise', 'bit operation', 'bit shift', 'XOR', 'AND, OR, NOT'],
    'gravity': ['gravitational', 'gravity', 'celestial', 'planet', 'gravitational constant'],
    'unit_conv': ['unit conversion', 'convert the following measurement', 'secret unit'],
    'cipher': ['encryption', 'cipher', 'encrypt', 'decrypt', 'encoded', 'secret code'],
    'numeral': ['numeral system', 'Roman numeral', 'ancient numeral', 'number system'],
    'symbol': ['symbol', 'symbolic', 'equation', 'transformation rule', 'symbol manipulation'],
}

def _classify_type(prompt):
    p_lower = prompt.lower()
    for t, kws in _type_keywords.items():
        for kw in kws:
            if kw.lower() in p_lower:
                return t
    return "unknown"

grpo_df = grpo_df.with_columns(
    pl.col("prompt").map_elements(_classify_type, return_dtype=pl.Utf8).alias("type")
)
print("Type distribution in full dataset:")
print(grpo_df.group_by("type").len().sort("type"))

# Strategic sampling: focus on improvable types, skip near-perfect ones
type_quotas = {
    "numeral": 50,        # already near-perfect, minimal GRPO
    "gravity": 200,       # decent headroom
    "unit_conv": 200,     # decent headroom
    "cipher": 200,        # biggest opportunity
    "bit_ops": 175,       # hardest but RL can discover patterns
    "symbol": 175,        # hard, RL exploration may help
}
grpo_frames = []
for t, n in type_quotas.items():
    subset = grpo_df.filter(pl.col("type") == t)
    actual_n = min(n, len(subset))
    grpo_frames.append(subset.sample(n=actual_n, seed=42))
grpo_df = pl.concat(grpo_frames).sample(fraction=1.0, seed=42)  # shuffle

print(f"\nGRPO dataset: {len(grpo_df)} samples")
for t in type_quotas:
    cnt = len(grpo_df.filter(pl.col("type") == t))
    print(f"  {t}: {cnt}")

# Pre-format prompts with enable_thinking=True
def format_grpo_prompt(example):
    user_msg = example["prompt"] + METRIC_SUFFIX
    messages = [{"role": "user", "content": user_msg}]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True
        )
    except Exception:
        text = f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
    return {"prompt": text, "answer": example["answer"]}

grpo_dataset = Dataset.from_pandas(grpo_df.to_pandas())
grpo_dataset = grpo_dataset.map(format_grpo_prompt, remove_columns=[c for c in grpo_dataset.column_names if c not in ["prompt", "answer"]])
print(f"✓ GRPO prompts formatted with enable_thinking=True")
print(f"Example prompt (first 200 chars):\n{grpo_dataset[0]['prompt'][:200]}")

# --- Unified reward function ---
def extract_boxed_answer(text):
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    if matches:
        return matches[-1].strip()
    m = re.search(r'\\boxed\{([^}]*?)$', text)
    if m:
        return m.group(1).strip()
    return None

def answers_match(pred, gold):
    if pred is None:
        return False
    p, g = pred.strip().lower(), gold.strip().lower()
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) <= 1e-2
    except (ValueError, OverflowError):
        return False

_debug_count = 0

def reward_func(completions, answer, **kwargs):
    """Single unified reward: correct=+1.0, has_format_wrong=-0.5, no_format=-1.0"""
    global _debug_count
    rewards = []
    for comp, gold in zip(completions, answer):
        text = comp[0]["content"] if isinstance(comp, list) else str(comp)
        pred = extract_boxed_answer(text)

        if _debug_count < 8:
            print(f"[GRPO debug #{_debug_count}] pred={pred} | gold={gold} | text_tail=...{text[-150:]}")
            _debug_count += 1

        if pred is None:
            rewards.append(-1.0)  # no format at all
        elif answers_match(pred, gold):
            rewards.append(1.0)   # correct!
        else:
            rewards.append(-0.5)  # format ok but wrong answer
    return rewards

print("✓ GRPO data and reward function ready")

In [ ]:
# --- GRPO Training ---

# GRPOTrainer accesses model.warnings_issued before super().__init__ creates it
if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

grpo_config = GRPOConfig(
    output_dir="/kaggle/working/grpo",
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=GRPO_LR,
    num_generations=GRPO_NUM_GEN,
    max_completion_length=GRPO_MAX_COMPLETION,
    max_prompt_length=GRPO_MAX_PROMPT,
    temperature=0.7,             # lower than 1.0 for better quality generations
    beta=0.0,                    # no KL penalty → no ref model needed
    bf16=True,
    max_grad_norm=1.0,
    warmup_ratio=0.1,            # warm up to preserve SFT gains
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_func],
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

print(f"Starting GRPO training:")
print(f"  Samples: {len(grpo_dataset)}, Generations/prompt: {GRPO_NUM_GEN}")
print(f"  Max completion: {GRPO_MAX_COMPLETION}, Max prompt: {GRPO_MAX_PROMPT}")
print(f"  LR: {GRPO_LR}, Temperature: 0.7, Beta: 0.0")
print(f"  Estimated gradient updates: {len(grpo_dataset) // 4}")
grpo_trainer.train()
print("✓ GRPO training complete")


## Save and Package Submission

In [ ]:
# Save adapter weights and config (model = PeftModel, updated by both SFT + GRPO)
model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Package into submission.zip (required by competition)
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify it contains adapter_config.json (required)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct and is ready to submit!")